### **WP3.5 - neural network model comparators**

This notebook trains GPU-ready neural-network comparators using the selected WP3.1 matrix and optional hourly clinical sequence dataset.


##### **Notebook Outline**

- **neural setup:** load the selected matrix, labels, identifiers, and optional hourly sequence arrays.
- **tabular neural models:** train MLP, TabNet, and transformer-style comparators where dependencies are available.
- **sequence and hybrid models:** train LSTM-style sequence models and hybrid models using hourly clinical summaries where available.
- **output save:** save training histories, model states, predictions, threshold tables, and comparison workbooks.


In [1]:
#core imports and output folders
import sys
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

current_path = Path.cwd().resolve()
project_path = None
for candidate_path in [current_path] + list(current_path.parents):
    if (candidate_path / "code").exists():
        project_path = candidate_path
        break
if project_path is None:
    project_path = Path("/Users/ahthini/Desktop/DissProject")
output_path = project_path / "outputs"
wp3_1_output_path = output_path / "WP3_1"
wp3_5_output_path = output_path / "WP3_5"
wp3_5_csv_output_path = wp3_5_output_path / "csv_tables"
wp3_5_model_output_path = wp3_5_output_path / "models"
wp3_5_image_output_path = wp3_5_output_path / "images"
for folder_path in [wp3_5_output_path, wp3_5_csv_output_path, wp3_5_model_output_path, wp3_5_image_output_path]:
    folder_path.mkdir(parents=True, exist_ok=True)

matrix_suffix = "shap_guided_reduced_feature_set"
print("WP3.5 neural-network output folder:", wp3_5_output_path)
print("Primary matrix suffix:", matrix_suffix)


WP3.5 neural-network output folder: /scratch/DissProject/outputs/WP3_5
Primary matrix suffix: shap_guided_reduced_feature_set


## 3.5.1 Setup and Paths

The notebook expects WP3.1 to have already saved the reduced train/test matrices and labels. On a GPU remote lab, clone the repository, place the allowed WP3.1 output files in the same folder structure, install the required packages, and run this notebook.

## 3.5.2 Load Reduced Matrix

The WP3.1 matrices are sparse because they include one-hot encoded predictors. Neural networks need dense mini-batches, so this notebook keeps the full matrix sparse on disk/in memory and converts each mini-batch to dense arrays during training.

In [2]:
#load processed reduced matrices, labels, ids, and feature names
X_train = sparse.load_npz(wp3_1_output_path / f"t3_1_X_train_processed_{matrix_suffix}.npz").tocsr()
X_test = sparse.load_npz(wp3_1_output_path / f"t3_1_X_test_processed_{matrix_suffix}.npz").tocsr()
y_train = pd.read_csv(wp3_1_output_path / "t3_1_y_train.csv").iloc[:, 0].astype(int).to_numpy()
y_test = pd.read_csv(wp3_1_output_path / "t3_1_y_test.csv").iloc[:, 0].astype(int).to_numpy()
feature_names = pd.read_csv(wp3_1_output_path / f"t3_1_processed_feature_names_{matrix_suffix}.csv")["processed_feature_name"].astype(str).tolist()
train_ids = pd.read_csv(wp3_1_output_path / "t3_1_train_ids.csv")
test_ids = pd.read_csv(wp3_1_output_path / "t3_1_test_ids.csv")

#store the row indices used in the neural validation split so sequence tensors can be aligned exactly.
all_train_indices = np.arange(X_train.shape[0])
train_model_indices, val_model_indices = train_test_split(
    all_train_indices, test_size=0.15, random_state=42, stratify=y_train)
X_train_model = X_train[train_model_indices]
X_val_model = X_train[val_model_indices]
y_train_model = y_train[train_model_indices]
y_val_model = y_train[val_model_indices]

scale_pos_weight = (y_train_model == 0).sum() / max((y_train_model == 1).sum(), 1)
print("Train matrix:", X_train.shape)
print("Validation matrix:", X_val_model.shape)
print("Test matrix:", X_test.shape)
print("Positive class percent:", round(100 * y_train.mean(), 2))
print("Scale positive weight:", round(scale_pos_weight, 4))

Train matrix: (191320, 973)
Validation matrix: (28698, 973)
Test matrix: (47171, 973)
Positive class percent: 19.88
Scale positive weight: 4.0296


## 3.5.3 Shared Evaluation Helpers

The same threshold roles used elsewhere in WP3 are kept here: default 0.50, validation F1 threshold, and balanced precision-recall threshold. This makes neural-network outputs directly comparable with logistic, random forest, and boosting models.

In [3]:
#shared metric and threshold helpers
def find_thresholds(y_true, probability):
    thresholds = np.round(np.linspace(0.01, 0.99, 99), 4)
    rows = []
    #loop through each item in this feature block
    for threshold in thresholds:
        y_pred = (probability >= threshold).astype(int)
        rows.append({"threshold": threshold,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
            "specificity": recall_score(1 - y_true, 1 - y_pred, zero_division=0),
            "f1_score": f1_score(y_true, y_pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)})
    threshold_df = pd.DataFrame(rows)
    f1_threshold = float(threshold_df.sort_values(["f1_score", "balanced_accuracy"], ascending=False).iloc[0]["threshold"])
    threshold_df["precision_recall_gap"] = (threshold_df["precision"] - threshold_df["recall_sensitivity"]).abs()
    balanced_pr_threshold = float(threshold_df.sort_values(["precision_recall_gap", "f1_score"], ascending=[True, False]).iloc[0]["threshold"])
    return f1_threshold, balanced_pr_threshold, threshold_df

#evaluate probability model
def evaluate_probability_model(model_name, y_true, probability, threshold, training_time_seconds=np.nan,
        train_inference_time_seconds=np.nan, test_inference_time_seconds=np.nan):
    y_pred = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"model": model_name, "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc_average_precision": average_precision_score(y_true, probability),
        "brier_score": brier_score_loss(y_true, probability),
        "true_negatives": tn, "false_positives": fp, "false_negatives": fn, "true_positives": tp,
        "training_time_seconds": training_time_seconds,
        "train_inference_time_seconds": train_inference_time_seconds,
        "test_inference_time_seconds": test_inference_time_seconds,
        "test_inference_time_per_admission_ms": 1000 * test_inference_time_seconds / len(y_true) if len(y_true) > 0 else np.nan}

#save neural outputs
def save_neural_outputs(model_name, output_stem, train_probability, val_probability, test_probability,
        training_time_seconds, train_inference_time_seconds, test_inference_time_seconds):
    f1_threshold, balanced_pr_threshold, threshold_df = find_thresholds(y_val_model, val_probability)
    #save this table or artefact for later review
    threshold_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_validation_threshold_search.csv", index=False)
    thresholds = [("default", 0.50), ("validation_f1", f1_threshold), ("balanced_precision_recall", balanced_pr_threshold)]
    performance_rows = []
    for threshold_role, threshold in thresholds:
        row = evaluate_probability_model(model_name, y_test, test_probability, threshold,
            training_time_seconds, train_inference_time_seconds, test_inference_time_seconds)
        row["threshold_role"] = threshold_role
        performance_rows.append(row)
    performance_df = pd.DataFrame(performance_rows)
    performance_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_performance.csv", index=False)
    prediction_df = test_ids.copy()
    if "readmitted_30d" not in prediction_df.columns:
        prediction_df["readmitted_30d"] = y_test
    prediction_df["predicted_probability"] = test_probability
    prediction_df["predicted_class"] = (test_probability >= f1_threshold).astype(int)
    prediction_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_test_predictions.csv", index=False)
    print(model_name, "test PR-AUC:", round(average_precision_score(y_test, test_probability), 4),
          "test ROC-AUC:", round(roc_auc_score(y_test, test_probability), 4),
          "validation F1 threshold:", f1_threshold,
          "balanced PR threshold:", balanced_pr_threshold)
    return performance_df, threshold_df, prediction_df


## 3.5.4 PyTorch Setup

On the remote lab, this will use CUDA if available. On a local CPU-only machine, it will still run, but more slowly. The neural-network models are trained with weighted binary cross-entropy to account for the lower readmission prevalence.

In [4]:
#optional pytorch import and device selection
try:
    import torch
    import torch.nn as nn
    torch_available = True
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"))
    print("PyTorch version:", torch.__version__)
    print("Using device:", device)
except Exception as error:
    torch_available = False
    device = None
    print("PyTorch unavailable; neural-network models that depend on PyTorch will be skipped.")
    print(error)



PyTorch version: 2.3.1+cu121
Using device: cuda


In [5]:
#mini-batch utilities for sparse matrices
run_pytorch_mlp = True
run_tabtransformer_style_model = True
run_tabnet_model = True
run_hourly_clinical_sequence_models = True

batch_size = 1024
max_epochs = 60
patience = 8
learning_rate = 1e-3

#tabnet is kept optional because the first gpu run showed weaker performance than the simpler neural baselines.
tabtransformer_max_epochs = 35
tabtransformer_patience = 6


#define sparse minibatches
def sparse_minibatches(X_sparse, y_array=None, batch_size=1024, shuffle=False, seed=42, feature_indices=None):
    rng = np.random.default_rng(seed)
    indices = np.arange(X_sparse.shape[0])
    if shuffle:
        rng.shuffle(indices)
    #loop through each item in this feature block
    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]
        X_batch = X_sparse[batch_indices]
        if feature_indices is not None:
            X_batch = X_batch[:, feature_indices]
        X_batch = X_batch.toarray().astype(np.float32)
        if y_array is None:
            yield X_batch
        else:
            yield X_batch, y_array[batch_indices].astype(np.float32)


#predict torch model
def predict_torch_model(model, X_sparse, batch_size=2048, feature_indices=None):
    model.eval()
    probabilities = []
    with torch.no_grad():
        #loop through each item in this feature block
        for X_batch in sparse_minibatches(X_sparse, batch_size=batch_size, feature_indices=feature_indices):
            X_tensor = torch.from_numpy(X_batch).to(device)
            logits = model(X_tensor).view(-1)
            probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(probabilities)


#train torch binary model
def train_torch_binary_model(model, model_name, feature_indices=None, batch_size_override=None,
        max_epochs_override=None, patience_override=None, learning_rate_override=None, weight_decay_override=1e-4):
    if not torch_available:
        return None, None
    model = model.to(device)
    active_batch_size = batch_size if batch_size_override is None else int(batch_size_override)
    active_max_epochs = max_epochs if max_epochs_override is None else int(max_epochs_override)
    active_patience = patience if patience_override is None else int(patience_override)
    active_learning_rate = learning_rate if learning_rate_override is None else float(learning_rate_override)
    positive_weight = torch.tensor([scale_pos_weight], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=active_learning_rate, weight_decay=float(weight_decay_override))
    best_state = None
    best_val_ap = -np.inf
    stale_epochs = 0
    history_rows = []
    training_start = time.perf_counter()
    for epoch in range(1, active_max_epochs + 1):
        model.train()
        epoch_losses = []
        for X_batch, y_batch in sparse_minibatches(X_train_model, y_train_model, batch_size=active_batch_size,
                shuffle=True, seed=42 + epoch, feature_indices=feature_indices):
            X_tensor = torch.from_numpy(X_batch).to(device)
            y_tensor = torch.from_numpy(y_batch).to(device)
            optimizer.zero_grad()
            logits = model(X_tensor).view(-1)
            loss = criterion(logits, y_tensor)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu().item()))
        val_probability = predict_torch_model(model, X_val_model, feature_indices=feature_indices)
        val_ap = average_precision_score(y_val_model, val_probability)
        val_roc = roc_auc_score(y_val_model, val_probability)
        history_rows.append({"epoch": epoch, "training_loss": np.mean(epoch_losses),
            "validation_pr_auc_average_precision": val_ap, "validation_roc_auc": val_roc})
        print(model_name, "epoch", epoch, "loss", round(np.mean(epoch_losses), 4),
            "val PR-AUC", round(val_ap, 4), "val ROC-AUC", round(val_roc, 4))
        if val_ap > best_val_ap + 1e-5:
            best_val_ap = val_ap
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= active_patience:
                print("Early stopping after", epoch, "epochs.")
                break
    training_time_seconds = time.perf_counter() - training_start
    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(device)
    history_df = pd.DataFrame(history_rows)
    return model, history_df, training_time_seconds



## 3.5.5 PyTorch MLP Variants

This section trains a small set of regularised MLP variants. The aim is not exhaustive tuning; it checks whether a simple neural tabular model can match the boosted-tree models without adding much runtime.




In [6]:
#train several compact pytorch mlp comparators using sampled hyperparameter screening
mlp_performance_df = pd.DataFrame()
mlp_variant_summary_df = pd.DataFrame()
mlp_models = {}

if torch_available and run_pytorch_mlp:
    import inspect
    from sklearn.model_selection import ParameterSampler

    class ReadmissionMLP(nn.Module):
        #initialise model or helper state
        def __init__(self, input_dim, hidden_layers, dropout_rates):
            super().__init__()
            layers = []
            previous_dim = input_dim

            #loop through each item in this feature block
            for layer_number, (hidden_dim, dropout_rate) in enumerate(zip(hidden_layers, dropout_rates), start=1):
                layers.append(nn.Linear(previous_dim, hidden_dim))
                if layer_number <= 2:
                    layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout_rate))
                previous_dim = hidden_dim

            layers.append(nn.Linear(previous_dim, 1))
            self.network = nn.Sequential(*layers)

        #run the model forward pass
        def forward(self, x):
            return self.network(x)

    #screen around the early-stopping region that looked strongest in the first run
    mlp_architecture_options = [
        {"hidden_layers": [256, 64], "dropout_rates": [0.35, 0.20]},
        {"hidden_layers": [384, 96], "dropout_rates": [0.35, 0.20]},
        {"hidden_layers": [512, 128], "dropout_rates": [0.35, 0.20]},
        {"hidden_layers": [512, 256, 64], "dropout_rates": [0.35, 0.25, 0.15]},
        {"hidden_layers": [384, 192, 64], "dropout_rates": [0.40, 0.25, 0.15]},
        {"hidden_layers": [768, 256, 64], "dropout_rates": [0.40, 0.30, 0.20]},
    ]

    mlp_parameter_space = {
        "architecture": mlp_architecture_options,
        "weight_decay": [1e-4, 3e-4, 5e-4, 1e-3],
        "batch_size": [512, 1024, 2048],
        "max_epochs": [35, 45, 55],
        "patience": [5, 6, 7],
    }

    #increase this on the remote lab if the gpu run is quick enough
    mlp_screening_n_iter = 10
    mlp_variant_configs = list(ParameterSampler(mlp_parameter_space, n_iter=mlp_screening_n_iter, random_state=42))

    train_signature = inspect.signature(train_torch_binary_model)
    train_parameter_names = set(train_signature.parameters)

    mlp_performance_frames = []
    mlp_summary_rows = []
    best_mlp_validation_pr_auc = -np.inf
    best_mlp_outputs = None

    #loop through each item in this feature block
    for config_number, config in enumerate(mlp_variant_configs, start=1):
        architecture = config["architecture"]
        hidden_layers = architecture["hidden_layers"]
        dropout_rates = architecture["dropout_rates"]

        variant_name = (
            f"mlp_screen_{config_number:02d}_"
            f"{'-'.join(str(layer) for layer in hidden_layers)}_"
            f"wd{str(config['weight_decay']).replace('.', 'p').replace('-', '')}_"
            f"bs{config['batch_size']}"
        )
        model_name = f"PyTorch MLP - {variant_name}"
        output_stem = f"t3_5_pytorch_mlp_{variant_name}"

        print("Training", model_name)
        print({
            "hidden_layers": hidden_layers,
            "dropout_rates": dropout_rates,
            "weight_decay": config["weight_decay"],
            "batch_size": config["batch_size"],
            "max_epochs": config["max_epochs"],
            "patience": config["patience"],
        })

        mlp_model = ReadmissionMLP(X_train.shape[1], hidden_layers, dropout_rates)

        train_kwargs = {
            "batch_size_override": config["batch_size"],
            "max_epochs_override": config["max_epochs"],
            "patience_override": config["patience"],
            "weight_decay_override": config["weight_decay"],
        }
        train_kwargs = {key: value for key, value in train_kwargs.items() if key in train_parameter_names}

        mlp_model, mlp_history_df, mlp_training_time_seconds = train_torch_binary_model(
            mlp_model, model_name, **train_kwargs)

        mlp_history_df["variant_name"] = variant_name
        mlp_history_df["hidden_layers"] = str(hidden_layers)
        mlp_history_df["dropout_rates"] = str(dropout_rates)
        mlp_history_df["weight_decay"] = config["weight_decay"]
        mlp_history_df["batch_size"] = config["batch_size"]
        #save this table or artefact for later review
        mlp_history_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_training_history.csv", index=False)

        torch.save(mlp_model.state_dict(), wp3_5_model_output_path / f"{output_stem}_state_dict.pt")
        mlp_models[variant_name] = mlp_model

        mlp_train_start = time.perf_counter()
        mlp_train_probability = predict_torch_model(mlp_model, X_train)
        mlp_train_inference_time_seconds = time.perf_counter() - mlp_train_start

        mlp_val_probability = predict_torch_model(mlp_model, X_val_model)

        mlp_test_start = time.perf_counter()
        mlp_test_probability = predict_torch_model(mlp_model, X_test)
        mlp_test_inference_time_seconds = time.perf_counter() - mlp_test_start

        variant_performance_df, _, _ = save_neural_outputs(
            model_name, output_stem,
            mlp_train_probability, mlp_val_probability, mlp_test_probability,
            mlp_training_time_seconds,
            mlp_train_inference_time_seconds,
            mlp_test_inference_time_seconds)

        variant_performance_df["mlp_variant"] = variant_name
        variant_performance_df["hidden_layers"] = str(hidden_layers)
        variant_performance_df["dropout_rates"] = str(dropout_rates)
        variant_performance_df["weight_decay"] = config["weight_decay"]
        variant_performance_df["batch_size"] = config["batch_size"]
        mlp_performance_frames.append(variant_performance_df)

        best_history_row = mlp_history_df.sort_values(
            "validation_pr_auc_average_precision", ascending=False).iloc[0]

        best_variant_validation_pr_auc = best_history_row["validation_pr_auc_average_precision"]
        mlp_summary_rows.append({
            "variant_name": variant_name,
            "hidden_layers": str(hidden_layers),
            "dropout_rates": str(dropout_rates),
            "weight_decay": config["weight_decay"],
            "batch_size": config["batch_size"],
            "max_epochs": config["max_epochs"],
            "patience": config["patience"],
            "best_validation_epoch": int(best_history_row["epoch"]),
            "best_validation_pr_auc_average_precision": best_variant_validation_pr_auc,
            "best_validation_roc_auc": best_history_row["validation_roc_auc"],
            "training_time_seconds": mlp_training_time_seconds,
        })

        if best_variant_validation_pr_auc > best_mlp_validation_pr_auc:
            best_mlp_validation_pr_auc = best_variant_validation_pr_auc
            best_mlp_outputs = {"variant_name": variant_name, "model_name": model_name,
                "model": mlp_model, "history_df": mlp_history_df,
                "train_probability": mlp_train_probability, "val_probability": mlp_val_probability,
                "test_probability": mlp_test_probability,
                "training_time_seconds": mlp_training_time_seconds,
                "train_inference_time_seconds": mlp_train_inference_time_seconds,
                "test_inference_time_seconds": mlp_test_inference_time_seconds}

        print()

    if mlp_performance_frames:
        mlp_performance_df = pd.concat(mlp_performance_frames, ignore_index=True)
        mlp_variant_summary_df = pd.DataFrame(mlp_summary_rows).sort_values(
            "best_validation_pr_auc_average_precision", ascending=False).reset_index(drop=True)

        mlp_performance_df.to_csv(wp3_5_csv_output_path / "t3_5_pytorch_mlp_screening_performance.csv", index=False)
        mlp_variant_summary_df.to_csv(wp3_5_csv_output_path / "t3_5_pytorch_mlp_variant_summary.csv", index=False)

        #save the best screened variant again under the canonical filenames used by later notebooks
        mlp_performance_df, _, _ = save_neural_outputs(
            best_mlp_outputs["model_name"],
            "t3_5_pytorch_mlp",
            best_mlp_outputs["train_probability"],
            best_mlp_outputs["val_probability"],
            best_mlp_outputs["test_probability"],
            best_mlp_outputs["training_time_seconds"],
            best_mlp_outputs["train_inference_time_seconds"],
            best_mlp_outputs["test_inference_time_seconds"])
        best_mlp_outputs["history_df"].to_csv(
            wp3_5_csv_output_path / "t3_5_pytorch_mlp_training_history.csv", index=False)
        torch.save(best_mlp_outputs["model"].state_dict(),
            wp3_5_model_output_path / "t3_5_pytorch_mlp_state_dict.pt")

        print("MLP variants ranked by validation PR-AUC:")
        display(mlp_variant_summary_df.round(4))
        print("Best MLP variant:", best_mlp_outputs["variant_name"])
    else:
        print("No MLP variants were trained.")
else:
    print("PyTorch MLP skipped.")



Training PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048
{'hidden_layers': [768, 256, 64], 'dropout_rates': [0.4, 0.3, 0.2], 'weight_decay': 0.001, 'batch_size': 2048, 'max_epochs': 45, 'patience': 7}
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 1 loss 0.9118 val PR-AUC 0.5774 val ROC-AUC 0.79
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 2 loss 0.8757 val PR-AUC 0.5879 val ROC-AUC 0.7961
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 3 loss 0.8624 val PR-AUC 0.5898 val ROC-AUC 0.7965
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 4 loss 0.852 val PR-AUC 0.5928 val ROC-AUC 0.799
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 5 loss 0.8413 val PR-AUC 0.5935 val ROC-AUC 0.7997
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 6 loss 0.8353 val PR-AUC 0.5943 val ROC-AUC 0.7993
PyTorch MLP - mlp_screen_01_768-256-64_wd0p001_bs2048 epoch 7 loss 0.8266 val PR-AUC 0.5944 val ROC-AUC 0.7974
PyTorch MLP - mlp_sc

,variant_name,hidden_layers,dropout_rates,weight_decay,batch_size,max_epochs,patience,best_validation_epoch,best_validation_pr_auc_average_precision,best_validation_roc_auc,training_time_seconds
0,mlp_screen_06_384-192-64_wd0p0001_bs512,"[384, 192, 64]","[0.4, 0.25, 0.15]",0.0001,512,45,6,4,0.5949,0.7997,7.4692
1,mlp_screen_04_512-256-64_wd0p0003_bs2048,"[512, 256, 64]","[0.35, 0.25, 0.15]",0.0003,2048,55,7,8,0.5947,0.7985,11.9225
2,mlp_screen_03_768-256-64_wd0p0003_bs1024,"[768, 256, 64]","[0.4, 0.3, 0.2]",0.0003,1024,45,6,5,0.5946,0.8000,8.3490
3,mlp_screen_01_768-256-64_wd0p001_bs2048,"[768, 256, 64]","[0.4, 0.3, 0.2]",0.0010,2048,45,7,7,0.5944,0.7974,13.0840
4,mlp_screen_10_512-128_wd0p0003_bs1024,"[512, 128]","[0.35, 0.2]",0.0003,1024,55,5,5,0.5932,0.8001,7.3193
5,mlp_screen_08_384-96_wd0p001_bs512,"[384, 96]","[0.35, 0.2]",0.0010,512,45,7,4,0.5929,0.7998,8.2794
6,mlp_screen_02_512-128_wd0p0001_bs512,"[512, 128]","[0.35, 0.2]",0.0001,512,35,6,4,0.5924,0.7991,7.3454
7,mlp_screen_09_512-128_wd0p001_bs512,"[512, 128]","[0.35, 0.2]",0.0010,512,45,5,7,0.5924,0.7977,8.6509
8,mlp_screen_07_384-96_wd0p0003_bs2048,"[384, 96]","[0.35, 0.2]",0.0003,2048,35,5,8,0.5919,0.7976,10.5823
9,mlp_screen_05_256-64_wd0p0001_bs2048,"[256, 64]","[0.35, 0.2]",0.0001,2048,35,5,9,0.5915,0.7969,10.9462


Best MLP variant: mlp_screen_06_384-192-64_wd0p0001_bs512


## 3.5.6 TabTransformer-Style Processed Feature Model

A classic TabTransformer works best with raw categorical embeddings. Your modelling pipeline currently uses processed sparse predictors, so this section uses a transformer-style attention model over the highest-priority processed features. It is included as a neural comparator rather than as the main recommended model.

In [7]:
#select top processed features for transformer-style model to keep attention memory manageable
max_transformer_features = 200
shap_importance_candidates = [
    output_path / "WP3_rapid_trial" / "tables" / "rapid_trial_shap_global_importance.xlsx",
    output_path / "WP4" / "csv_tables" / "t4_shap_global_importance.csv",
    output_path / "WP4" / "t4_shap_explanation_tables.xlsx"]

selected_transformer_feature_names = []
for candidate_path in shap_importance_candidates:
    if candidate_path.exists():
        try:
            if candidate_path.suffix == ".xlsx":
                temp_df = pd.read_excel(candidate_path)
            else:
                temp_df = pd.read_csv(candidate_path)
            if "processed_feature_name" in temp_df.columns:
                score_col = next((col for col in ["mean_abs_shap", "importance", "feature_importance"] if col in temp_df.columns), None)
                if score_col is not None:
                    temp_df = temp_df.sort_values(score_col, ascending=False)
                    selected_transformer_feature_names = [feature for feature in temp_df["processed_feature_name"].astype(str).tolist()
                        if feature in feature_names][:max_transformer_features]
                    break
        except Exception:
            pass
if not selected_transformer_feature_names:
    nonzero_counts = np.asarray(X_train.getnnz(axis=0)).ravel()
    top_indices = np.argsort(nonzero_counts)[::-1][:max_transformer_features]
    selected_transformer_feature_names = [feature_names[index] for index in top_indices]
transformer_feature_indices = np.array([feature_names.index(feature) for feature in selected_transformer_feature_names], dtype=int)
pd.DataFrame({"processed_feature_name": selected_transformer_feature_names}).to_csv(
    wp3_5_csv_output_path / "t3_5_tabtransformer_style_selected_features.csv", index=False)
print("Transformer-style feature count:", len(transformer_feature_indices))


Transformer-style feature count: 200


In [8]:
#train transformer-style comparators on selected processed features using sampled hyperparameter screening
tabtransformer_performance_df = pd.DataFrame()
tabtransformer_variant_summary_df = pd.DataFrame()
tabtransformer_models = {}

if torch_available and run_tabtransformer_style_model:
    from sklearn.model_selection import ParameterSampler

    class ProcessedFeatureTransformer(nn.Module):
        #initialise model or helper state
        def __init__(self, input_dim, d_model=32, n_heads=4, n_layers=2, dim_feedforward=128, dropout=0.20):
            super().__init__()
            self.feature_embedding = nn.Parameter(torch.randn(input_dim, d_model) * 0.02)
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

            encoder_layer = nn.TransformerEncoderLayer(d_model=d_model,
                nhead=n_heads, dim_feedforward=dim_feedforward,
                dropout=dropout, batch_first=True, activation="gelu")

            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
            self.head = nn.Sequential(nn.LayerNorm(d_model),
                nn.Dropout(dropout), nn.Linear(d_model, 1))

        #run the model forward pass
        def forward(self, x):
            tokens = x.unsqueeze(-1) * self.feature_embedding.unsqueeze(0)
            cls = self.cls_token.expand(x.shape[0], -1, -1)
            encoded = self.encoder(torch.cat([cls, tokens], dim=1))
            return self.head(encoded[:, 0, :])

    #keep only valid d_model/head combinations and avoid very large models
    tabtransformer_architecture_options = [
        {"d_model": 16, "n_heads": 2, "n_layers": 1, "dim_feedforward": 64},
        {"d_model": 24, "n_heads": 3, "n_layers": 1, "dim_feedforward": 96},
        {"d_model": 32, "n_heads": 4, "n_layers": 1, "dim_feedforward": 128},
        {"d_model": 32, "n_heads": 4, "n_layers": 2, "dim_feedforward": 128},
        {"d_model": 48, "n_heads": 4, "n_layers": 1, "dim_feedforward": 192},
        {"d_model": 48, "n_heads": 6, "n_layers": 1, "dim_feedforward": 192},]

    tabtransformer_parameter_space = {"architecture": tabtransformer_architecture_options,
        "dropout": [0.20, 0.30, 0.40], "weight_decay": [1e-4, 2e-4, 5e-4, 1e-3],
        "batch_size": [128, 256], "max_epochs": [25, 35, 45], "patience": [4, 5, 6],}

    #increase this if the gpu run is quick enough
    tabtransformer_screening_n_iter = 8
    tabtransformer_variant_configs = list(ParameterSampler(
        tabtransformer_parameter_space, n_iter=tabtransformer_screening_n_iter, random_state=42))

    tabtransformer_performance_frames = []
    tabtransformer_summary_rows = []
    best_tabtransformer_validation_pr_auc = -np.inf
    best_tabtransformer_outputs = None

    #loop through each item in this feature block
    for config_number, config in enumerate(tabtransformer_variant_configs, start=1):
        architecture = config["architecture"]

        variant_name = (f"tabtransformer_screen_{config_number:02d}_"
            f"d{architecture['d_model']}_h{architecture['n_heads']}_"
            f"l{architecture['n_layers']}_"
            f"drop{str(config['dropout']).replace('.', 'p')}_"
            f"wd{str(config['weight_decay']).replace('.', 'p').replace('-', '')}")

        model_name = f"Processed-feature TabTransformer - {variant_name}"
        output_stem = f"t3_5_tabtransformer_style_{variant_name}"

        print("Training", model_name)
        print({"d_model": architecture["d_model"],
            "n_heads": architecture["n_heads"],
            "n_layers": architecture["n_layers"],
            "dim_feedforward": architecture["dim_feedforward"],
            "dropout": config["dropout"],
            "weight_decay": config["weight_decay"],
            "batch_size": config["batch_size"],
            "max_epochs": config["max_epochs"],
            "patience": config["patience"],})

        tabtransformer_model = ProcessedFeatureTransformer(
            len(transformer_feature_indices),
            d_model=architecture["d_model"],
            n_heads=architecture["n_heads"],
            n_layers=architecture["n_layers"],
            dim_feedforward=architecture["dim_feedforward"],
            dropout=config["dropout"])

        tabtransformer_model, tabtransformer_history_df, tabtransformer_training_time_seconds = train_torch_binary_model(
            tabtransformer_model, model_name,
            feature_indices=transformer_feature_indices,
            batch_size_override=config["batch_size"],
            max_epochs_override=config["max_epochs"],
            patience_override=config["patience"],
            weight_decay_override=config["weight_decay"])

        tabtransformer_history_df["variant_name"] = variant_name
        tabtransformer_history_df["d_model"] = architecture["d_model"]
        tabtransformer_history_df["n_heads"] = architecture["n_heads"]
        tabtransformer_history_df["n_layers"] = architecture["n_layers"]
        tabtransformer_history_df["dropout"] = config["dropout"]
        tabtransformer_history_df["weight_decay"] = config["weight_decay"]
        tabtransformer_history_df["batch_size"] = config["batch_size"]
        tabtransformer_history_df.to_csv(
            wp3_5_csv_output_path / f"{output_stem}_training_history.csv", index=False)

        torch.save(tabtransformer_model.state_dict(),
            wp3_5_model_output_path / f"{output_stem}_state_dict.pt")

        tabtransformer_models[variant_name] = tabtransformer_model

        tab_train_start = time.perf_counter()
        tabtransformer_train_probability = predict_torch_model(
            tabtransformer_model, X_train, batch_size=256, feature_indices=transformer_feature_indices)
        tabtransformer_train_inference_time_seconds = time.perf_counter() - tab_train_start

        tabtransformer_val_probability = predict_torch_model(
            tabtransformer_model, X_val_model, batch_size=256, feature_indices=transformer_feature_indices)

        tab_test_start = time.perf_counter()
        tabtransformer_test_probability = predict_torch_model(
            tabtransformer_model, X_test, batch_size=256, feature_indices=transformer_feature_indices)
        tabtransformer_test_inference_time_seconds = time.perf_counter() - tab_test_start

        variant_performance_df, variant_prediction_df, variant_threshold_df = save_neural_outputs(
            model_name, output_stem, tabtransformer_train_probability,
            tabtransformer_val_probability, tabtransformer_test_probability,
            tabtransformer_training_time_seconds,
            tabtransformer_train_inference_time_seconds,
            tabtransformer_test_inference_time_seconds)

        variant_performance_df["tabtransformer_variant"] = variant_name
        variant_performance_df["d_model"] = architecture["d_model"]
        variant_performance_df["n_heads"] = architecture["n_heads"]
        variant_performance_df["n_layers"] = architecture["n_layers"]
        variant_performance_df["dropout"] = config["dropout"]
        variant_performance_df["weight_decay"] = config["weight_decay"]
        variant_performance_df["batch_size"] = config["batch_size"]
        tabtransformer_performance_frames.append(variant_performance_df)

        best_history_row = tabtransformer_history_df.sort_values(
            "validation_pr_auc_average_precision", ascending=False).iloc[0]

        best_variant_validation_pr_auc = best_history_row["validation_pr_auc_average_precision"]

        tabtransformer_summary_rows.append({"variant_name": variant_name,
            "d_model": architecture["d_model"], "n_heads": architecture["n_heads"],
            "n_layers": architecture["n_layers"], "dim_feedforward": architecture["dim_feedforward"],
            "dropout": config["dropout"], "weight_decay": config["weight_decay"],
            "batch_size": config["batch_size"], "max_epochs": config["max_epochs"],
            "patience": config["patience"], "best_validation_epoch": int(best_history_row["epoch"]),
            "best_validation_pr_auc_average_precision": best_variant_validation_pr_auc,
            "best_validation_roc_auc": best_history_row["validation_roc_auc"],
            "training_time_seconds": tabtransformer_training_time_seconds,})

        if best_variant_validation_pr_auc > best_tabtransformer_validation_pr_auc:
            best_tabtransformer_validation_pr_auc = best_variant_validation_pr_auc
            best_tabtransformer_outputs = {"model": tabtransformer_model,
                "model_name": model_name, "variant_name": variant_name,
                "history_df": tabtransformer_history_df.copy(),
                "train_probability": tabtransformer_train_probability,
                "val_probability": tabtransformer_val_probability,
                "test_probability": tabtransformer_test_probability,
                "training_time_seconds": tabtransformer_training_time_seconds,
                "train_inference_time_seconds": tabtransformer_train_inference_time_seconds,
                "test_inference_time_seconds": tabtransformer_test_inference_time_seconds,}

        print()

    if tabtransformer_performance_frames:
        tabtransformer_performance_df = pd.concat(tabtransformer_performance_frames, ignore_index=True)
        tabtransformer_variant_summary_df = pd.DataFrame(tabtransformer_summary_rows).sort_values(
            "best_validation_pr_auc_average_precision", ascending=False).reset_index(drop=True)

        tabtransformer_performance_df.to_csv(
            wp3_5_csv_output_path / "t3_5_tabtransformer_style_screening_performance.csv", index=False)
        tabtransformer_variant_summary_df.to_csv(
            wp3_5_csv_output_path / "t3_5_tabtransformer_style_variant_summary.csv", index=False)

        #save the best screened variant again under the canonical filenames used by later notebooks
        tabtransformer_performance_df, _, _ = save_neural_outputs(
            best_tabtransformer_outputs["model_name"],
            "t3_5_tabtransformer_style",
            best_tabtransformer_outputs["train_probability"],
            best_tabtransformer_outputs["val_probability"],
            best_tabtransformer_outputs["test_probability"],
            best_tabtransformer_outputs["training_time_seconds"],
            best_tabtransformer_outputs["train_inference_time_seconds"],
            best_tabtransformer_outputs["test_inference_time_seconds"])

        best_tabtransformer_outputs["history_df"].to_csv(
            wp3_5_csv_output_path / "t3_5_tabtransformer_style_training_history.csv", index=False)
        torch.save(best_tabtransformer_outputs["model"].state_dict(),
            wp3_5_model_output_path / "t3_5_tabtransformer_style_state_dict.pt")

        print("TabTransformer-style variants ranked by validation PR-AUC:")
        display(tabtransformer_variant_summary_df.round(4))
        print("Best TabTransformer-style variant:", best_tabtransformer_outputs["variant_name"])
    else:
        print("No TabTransformer-style variants were trained.")
else:
    print("Processed-feature TabTransformer skipped.")

Training Processed-feature TabTransformer - tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0p0005
{'d_model': 48, 'n_heads': 6, 'n_layers': 1, 'dim_feedforward': 192, 'dropout': 0.3, 'weight_decay': 0.0005, 'batch_size': 128, 'max_epochs': 25, 'patience': 6}
Processed-feature TabTransformer - tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0p0005 epoch 1 loss 0.9368 val PR-AUC 0.5571 val ROC-AUC 0.7823
Processed-feature TabTransformer - tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0p0005 epoch 2 loss 0.9059 val PR-AUC 0.5609 val ROC-AUC 0.7868
Processed-feature TabTransformer - tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0p0005 epoch 3 loss 0.9008 val PR-AUC 0.5605 val ROC-AUC 0.7883
Processed-feature TabTransformer - tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0p0005 epoch 4 loss 0.8974 val PR-AUC 0.567 val ROC-AUC 0.7901
Processed-feature TabTransformer - tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0p0005 epoch 5 loss 0.8954 val PR-AUC 0.5661 val ROC-AUC 0.7901
Processed-feature TabTra

,variant_name,d_model,n_heads,n_layers,dim_feedforward,dropout,weight_decay,batch_size,max_epochs,patience,best_validation_epoch,best_validation_pr_auc_average_precision,best_validation_roc_auc,training_time_seconds
0,tabtransformer_screen_08_d32_h4_l1_drop0p2_wd0...,32,4,1,128,0.2,0.0005,128,45,6,33,0.5870,0.7965,404.2513
1,tabtransformer_screen_02_d32_h4_l2_drop0p4_wd0...,32,4,2,128,0.4,0.0001,256,45,6,19,0.5866,0.7955,486.5091
2,tabtransformer_screen_05_d48_h6_l1_drop0p2_wd0...,48,6,1,192,0.2,0.0010,128,35,4,22,0.5859,0.7951,387.5101
3,tabtransformer_screen_01_d48_h6_l1_drop0p3_wd0...,48,6,1,192,0.3,0.0005,128,25,6,25,0.5855,0.7956,369.1681
4,tabtransformer_screen_03_d48_h6_l1_drop0p4_wd0...,48,6,1,192,0.4,0.0005,256,45,6,45,0.5847,0.7947,646.2614
5,tabtransformer_screen_04_d48_h6_l1_drop0p3_wd0...,48,6,1,192,0.3,0.0005,128,35,4,17,0.5828,0.7942,312.9633
6,tabtransformer_screen_07_d16_h2_l1_drop0p2_wd0...,16,2,1,64,0.2,0.0002,256,35,4,17,0.5737,0.7903,121.7743
7,tabtransformer_screen_06_d48_h4_l1_drop0p4_wd0...,48,4,1,192,0.4,0.0001,256,25,4,9,0.5640,0.7895,152.6590


Best TabTransformer-style variant: tabtransformer_screen_08_d32_h4_l1_drop0p2_wd0p0005


## 3.5.7 Optional TabNet

TabNet is included as an optional comparator because it is a recognised neural architecture for tabular data. It requires the `pytorch-tabnet` package. If the package is unavailable, this section skips cleanly and the PyTorch MLP remains the main neural-network comparator.

In [9]:
#optional tabnet comparator using sampled hyperparameter screening
#tabnet is slower than the mlp, so the search is intentionally compact and uses validation pr-auc for final ranking.
tabnet_performance_df = pd.DataFrame()
tabnet_variant_summary_df = pd.DataFrame()
tabnet_models = {}

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    from sklearn.model_selection import ParameterSampler
    tabnet_available = True
except Exception as error:
    tabnet_available = False
    print("TabNet unavailable. Install with: pip install pytorch-tabnet")
    print(error)

if tabnet_available and run_tabnet_model:
    X_train_tabnet = X_train_model.toarray().astype(np.float32)
    X_val_tabnet = X_val_model.toarray().astype(np.float32)
    X_test_tabnet = X_test.toarray().astype(np.float32)
    X_train_full_tabnet = X_train.toarray().astype(np.float32)

    tabnet_parameter_space = {
        "n_d": [16, 24, 32],
        "n_a": [16, 24, 32],
        "n_steps": [3, 4, 5],
        "gamma": [1.2, 1.4, 1.6],
        "lambda_sparse": [1e-5, 1e-4, 5e-4],
        "learning_rate": [1e-3, 2e-3, 3e-3],
        "batch_size": [1024, 2048, 4096],
        "virtual_batch_size": [128, 256],
        "max_epochs": [80, 100, 120],
        "patience": [10, 12, 15],
        "positive_weight_multiplier": [0.75, 1.0, 1.25],
    }

    #increase this on the remote lab if tabnet is still running comfortably
    tabnet_screening_n_iter = 8
    tabnet_variant_configs = list(ParameterSampler(tabnet_parameter_space, n_iter=tabnet_screening_n_iter, random_state=42))

    tabnet_performance_frames = []
    tabnet_summary_rows = []
    best_tabnet_validation_pr_auc = -np.inf
    best_tabnet_outputs = None

    #loop through each item in this feature block
    for config_number, config in enumerate(tabnet_variant_configs, start=1):
        if config["n_d"] != config["n_a"]:
            continue

        variant_name = (
            f"tabnet_screen_{config_number:02d}_"
            f"d{config['n_d']}_steps{config['n_steps']}_"
            f"gamma{str(config['gamma']).replace('.', 'p')}_"
            f"lr{str(config['learning_rate']).replace('.', 'p').replace('-', '')}"
        )
        model_name = f"TabNet - {variant_name}"
        output_stem = f"t3_5_tabnet_{variant_name}"

        print("Training", model_name)
        print(config)

        adjusted_positive_weight = float(scale_pos_weight) * config["positive_weight_multiplier"]
        tabnet_model = TabNetClassifier(
            n_d=config["n_d"],
            n_a=config["n_a"],
            n_steps=config["n_steps"],
            gamma=config["gamma"],
            lambda_sparse=config["lambda_sparse"],
            optimizer_params={"lr": config["learning_rate"]},
            seed=42,
            verbose=10,
            device_name="cuda" if torch_available and torch.cuda.is_available() else "cpu")

        tabnet_start = time.perf_counter()
        tabnet_model.fit(
            X_train_tabnet,
            y_train_model,
            eval_set=[(X_val_tabnet, y_val_model)],
            eval_name=["validation"],
            eval_metric=["auc"],
            max_epochs=config["max_epochs"],
            patience=config["patience"],
            batch_size=config["batch_size"],
            virtual_batch_size=config["virtual_batch_size"],
            weights={0: 1.0, 1: adjusted_positive_weight})
        tabnet_training_time_seconds = time.perf_counter() - tabnet_start

        tabnet_model.save_model(str(wp3_5_model_output_path / f"{output_stem}_model"))
        tabnet_models[variant_name] = tabnet_model

        tabnet_train_start = time.perf_counter()
        #score admissions as predicted readmission probabilities
        tabnet_train_probability = tabnet_model.predict_proba(X_train_full_tabnet)[:, 1]
        tabnet_train_inference_time_seconds = time.perf_counter() - tabnet_train_start

        tabnet_val_probability = tabnet_model.predict_proba(X_val_tabnet)[:, 1]

        tabnet_test_start = time.perf_counter()
        tabnet_test_probability = tabnet_model.predict_proba(X_test_tabnet)[:, 1]
        tabnet_test_inference_time_seconds = time.perf_counter() - tabnet_test_start

        variant_performance_df, variant_prediction_df, variant_threshold_df = save_neural_outputs(
            model_name,
            output_stem,
            tabnet_train_probability,
            tabnet_val_probability,
            tabnet_test_probability,
            tabnet_training_time_seconds,
            tabnet_train_inference_time_seconds,
            tabnet_test_inference_time_seconds)

        validation_pr_auc = average_precision_score(y_val_model, tabnet_val_probability)
        validation_roc_auc = roc_auc_score(y_val_model, tabnet_val_probability)

        #loop through each item in this feature block
        for key, value in config.items():
            variant_performance_df[key] = value
        variant_performance_df["tabnet_variant"] = variant_name
        variant_performance_df["effective_positive_class_weight"] = adjusted_positive_weight
        tabnet_performance_frames.append(variant_performance_df)

        tabnet_summary_rows.append({
            "variant_name": variant_name,
            **config,
            "effective_positive_class_weight": adjusted_positive_weight,
            "validation_pr_auc_average_precision": validation_pr_auc,
            "validation_roc_auc": validation_roc_auc,
            "training_time_seconds": tabnet_training_time_seconds,
        })

        if validation_pr_auc > best_tabnet_validation_pr_auc:
            best_tabnet_validation_pr_auc = validation_pr_auc
            best_tabnet_outputs = {
                "model": tabnet_model,
                "model_name": model_name,
                "variant_name": variant_name,
                "train_probability": tabnet_train_probability,
                "val_probability": tabnet_val_probability,
                "test_probability": tabnet_test_probability,
                "training_time_seconds": tabnet_training_time_seconds,
                "train_inference_time_seconds": tabnet_train_inference_time_seconds,
                "test_inference_time_seconds": tabnet_test_inference_time_seconds,
            }

        print("Validation PR-AUC:", round(validation_pr_auc, 4))
        print("Validation ROC-AUC:", round(validation_roc_auc, 4))
        print()

    if tabnet_performance_frames:
        tabnet_screening_performance_df = pd.concat(tabnet_performance_frames, ignore_index=True)
        tabnet_variant_summary_df = pd.DataFrame(tabnet_summary_rows).sort_values(
            "validation_pr_auc_average_precision", ascending=False).reset_index(drop=True)

        tabnet_screening_performance_df.to_csv(
            wp3_5_csv_output_path / "t3_5_tabnet_screening_performance.csv", index=False)
        tabnet_variant_summary_df.to_csv(
            wp3_5_csv_output_path / "t3_5_tabnet_variant_summary.csv", index=False)

        #save the best screened variant again under the canonical filenames used by later notebooks
        tabnet_performance_df, _, _ = save_neural_outputs(
            best_tabnet_outputs["model_name"],
            "t3_5_tabnet",
            best_tabnet_outputs["train_probability"],
            best_tabnet_outputs["val_probability"],
            best_tabnet_outputs["test_probability"],
            best_tabnet_outputs["training_time_seconds"],
            best_tabnet_outputs["train_inference_time_seconds"],
            best_tabnet_outputs["test_inference_time_seconds"])
        best_tabnet_outputs["model"].save_model(str(wp3_5_model_output_path / "t3_5_tabnet_model"))

        print("TabNet variants ranked by validation PR-AUC:")
        display(tabnet_variant_summary_df.round(4))
        print("Best TabNet variant:", best_tabnet_outputs["variant_name"])
    else:
        print("No TabNet variants were trained.")
else:
    print("TabNet skipped.")



Training TabNet - tabnet_screen_01_d24_steps4_gamma1p4_lr0p002
{'virtual_batch_size': 256, 'positive_weight_multiplier': 1.0, 'patience': 12, 'n_steps': 4, 'n_d': 24, 'n_a': 24, 'max_epochs': 120, 'learning_rate': 0.002, 'lambda_sparse': 1e-05, 'gamma': 1.4, 'batch_size': 1024}
epoch 0  | loss: 0.84698 | validation_auc: 0.56388 |  0:00:02s
epoch 10 | loss: 0.59326 | validation_auc: 0.74339 |  0:00:31s
epoch 20 | loss: 0.57672 | validation_auc: 0.76215 |  0:00:59s
epoch 30 | loss: 0.55163 | validation_auc: 0.78224 |  0:01:27s
epoch 40 | loss: 0.53336 | validation_auc: 0.79027 |  0:01:55s
epoch 50 | loss: 0.51574 | validation_auc: 0.78559 |  0:02:24s

Early stopping occurred at epoch 51 with best_epoch = 39 and best_validation_auc = 0.79048
Successfully saved model at /scratch/DissProject/outputs/WP3_5/models/t3_5_tabnet_tabnet_screen_01_d24_steps4_gamma1p4_lr0p002_model.zip
TabNet - tabnet_screen_01_d24_steps4_gamma1p4_lr0p002 test PR-AUC: 0.5552 test ROC-AUC: 0.7789 validation F1 thres

,variant_name,virtual_batch_size,positive_weight_multiplier,patience,n_steps,n_d,n_a,max_epochs,learning_rate,lambda_sparse,gamma,batch_size,effective_positive_class_weight,validation_pr_auc_average_precision,validation_roc_auc,training_time_seconds
0,tabnet_screen_05_d16_steps4_gamma1p6_lr0p002,128,1.00,15,4,16,16,120,0.002,0.0005,1.6,2048,4.0296,0.5830,0.7930,377.8978
1,tabnet_screen_08_d24_steps5_gamma1p2_lr0p003,128,0.75,10,5,24,24,100,0.003,0.0000,1.2,4096,3.0222,0.5730,0.7865,348.0765
2,tabnet_screen_01_d24_steps4_gamma1p4_lr0p002,256,1.00,12,4,24,24,120,0.002,0.0000,1.4,1024,4.0296,0.5700,0.7905,206.0059
3,tabnet_screen_07_d32_steps3_gamma1p2_lr0p002,256,0.75,10,3,32,32,80,0.002,0.0001,1.2,1024,3.0222,0.5693,0.7824,110.1695
4,tabnet_screen_04_d32_steps3_gamma1p6_lr0p001,128,0.75,10,3,32,32,100,0.001,0.0001,1.6,4096,3.0222,0.5688,0.7757,236.0741
5,tabnet_screen_06_d32_steps4_gamma1p4_lr0p002,128,1.25,10,4,32,32,100,0.002,0.0000,1.4,2048,5.0370,0.5660,0.7863,238.6119


Best TabNet variant: tabnet_screen_05_d16_steps4_gamma1p6_lr0p002


## 3.5.8 Sequential Clinical-Timeline Neural Network Sensitivity Analysis

The hourly clinical sequence is available for only a subset of admissions, so these models are treated as sequence-subset sensitivity analyses. A tabular-only MLP is trained on the same subset before the GRU and LSTM models so the notebook can test whether the hourly sequence adds information beyond the tabular matrix.





In [10]:
#build first-72-hour clinical sequence tensors aligned to the wp3.1 train/validation/test split
hourly_sequence_performance_df = pd.DataFrame()
hybrid_sequence_performance_df = pd.DataFrame()
sequence_output_path = wp3_5_output_path / "sequence_arrays"
sequence_output_path.mkdir(parents=True, exist_ok=True)
rich_hourly_sequence_file = output_path / "t2_3_hourly_clinical_sequence_dataset.csv"
sequence_max_hours = 72
sequence_model_ready = False

base_sequence_columns = ["heart_rate", "mean_bp_arterial", "mean_bp_noninvasive", "respiratory_rate",
    "spo2", "systolic_bp_arterial", "systolic_bp_noninvasive"]
event_sequence_columns = [
    "num_poe_orders", "num_poe_medication_orders", "num_poe_lab_orders", "num_poe_imaging_orders",
    "num_poe_consult_orders", "num_poe_safety_orders", "num_poe_followup_aftercare_orders",
    "num_poe_discharge_admin_orders", "num_poe_discontinued_or_cancelled_orders", "num_unique_poe_order_types_hourly",
    "num_lab_events", "num_abnormal_lab_events", "num_priority_lab_events", "num_numeric_lab_results",
    "num_emar_events", "num_emar_administered_events", "num_emar_not_given_events", "num_emar_delayed_events",
    "num_emar_started_or_stopped_events", "num_psychotropic_emar_events",
    "num_icu_input_events", "num_icu_output_events", "num_icu_procedure_events", "num_icu_datetime_events",
    "num_total_clinical_events", "num_order_lab_medication_events", "num_care_activity_events", "num_instability_process_events"]

if rich_hourly_sequence_file.exists():
    sequence_source_file = rich_hourly_sequence_file
    sequence_source_label = "rich_hourly_clinical_sequence"
    requested_sequence_columns = base_sequence_columns + event_sequence_columns
else:
    sequence_source_file = None
    sequence_source_label = "none"
    requested_sequence_columns = []

if not torch_available or not run_hourly_clinical_sequence_models:
    print("Hourly clinical sequence models skipped.")
elif sequence_source_file is None:
    print("Hourly clinical sequence models skipped because the WP2.3 clinical sequence file was not found:")
    print(rich_hourly_sequence_file)
else:
    #only columns needed for the sequence tensor are loaded to keep memory use manageable.
    available_header = pd.read_csv(sequence_source_file, nrows=0).columns.tolist()
    sequence_raw_columns = [col for col in requested_sequence_columns if col in available_header]
    hourly_use_columns = ["subject_id", "hadm_id", "hour_window"] + sequence_raw_columns
    hourly_sequence_df = pd.read_csv(sequence_source_file, usecols=hourly_use_columns)
    hourly_sequence_df = hourly_sequence_df[hourly_sequence_df["hour_window"].between(1, sequence_max_hours)].copy()
    hourly_sequence_df[sequence_raw_columns] = hourly_sequence_df[sequence_raw_columns].apply(pd.to_numeric, errors="coerce")

    event_columns_available = [col for col in event_sequence_columns if col in sequence_raw_columns]
    value_columns_available = [col for col in sequence_raw_columns if col not in event_columns_available]

    #event-count channels are true zeros when no event was recorded in that observed admission-hour.
    if event_columns_available:
        hourly_sequence_df[event_columns_available] = hourly_sequence_df[event_columns_available].fillna(0)

    #fit scalers only on training admissions to avoid using test-set distributional information.
    train_key_df = train_ids[["subject_id", "hadm_id"]].drop_duplicates().copy()
    #join the derived features back to the admission table
    hourly_train_rows = hourly_sequence_df.merge(train_key_df, on=["subject_id", "hadm_id"], how="inner")
    sequence_scaler = StandardScaler()
    #fit the model on the current training data
    sequence_scaler.fit(hourly_train_rows[sequence_raw_columns].fillna(0))

    #the tensor contains scaled values plus per-feature observed/missingness channels.
    sequence_feature_columns = sequence_raw_columns + [f"{col}_observed_mask" for col in sequence_raw_columns]

    #make clinical sequence tensor
    def make_clinical_sequence_tensor(id_frame, split_name):
        id_frame = id_frame[["subject_id", "hadm_id"]].drop_duplicates().reset_index(drop=True).copy()
        id_frame["sequence_row_index"] = np.arange(len(id_frame))
        #join the derived features back to the admission table
        temp = hourly_sequence_df.merge(id_frame, on=["subject_id", "hadm_id"], how="inner")
        temp = temp.sort_values(["sequence_row_index", "hour_window"])
        sequence_array = np.zeros((len(id_frame), sequence_max_hours, len(sequence_feature_columns)), dtype=np.float32)
        mask_array = np.zeros((len(id_frame), sequence_max_hours), dtype=np.float32)
        if not temp.empty:
            observed_feature_mask = temp[sequence_raw_columns].notna().to_numpy(dtype=np.float32)
            scaled_values = sequence_scaler.transform(temp[sequence_raw_columns].fillna(0)).astype(np.float32)
            scaled_values = np.nan_to_num(scaled_values, nan=0.0, posinf=0.0, neginf=0.0)
            sequence_values = np.concatenate([scaled_values, observed_feature_mask], axis=1)
            row_positions = temp["sequence_row_index"].to_numpy(dtype=int)
            hour_positions = temp["hour_window"].to_numpy(dtype=int) - 1
            sequence_array[row_positions, hour_positions, :] = sequence_values
            mask_array[row_positions, hour_positions] = (observed_feature_mask.sum(axis=1) > 0).astype(np.float32)
        sequence_available = mask_array.sum(axis=1) > 0
        np.savez_compressed(sequence_output_path / f"t3_5_{split_name}_first72h_clinical_sequence.npz",
            sequence_array=sequence_array, mask_array=mask_array, sequence_available=sequence_available,
            subject_id=id_frame["subject_id"].to_numpy(), hadm_id=id_frame["hadm_id"].to_numpy())
        return sequence_array, mask_array, sequence_available

    train_sequence_array, train_sequence_mask, train_sequence_available = make_clinical_sequence_tensor(train_ids, "train")
    test_sequence_array, test_sequence_mask, test_sequence_available = make_clinical_sequence_tensor(test_ids, "test")

    train_model_sequence_available = train_sequence_available[train_model_indices]
    val_model_sequence_available = train_sequence_available[val_model_indices]
    X_train_model_sequence = train_sequence_array[train_model_indices][train_model_sequence_available]
    X_val_model_sequence = train_sequence_array[val_model_indices][val_model_sequence_available]
    train_model_sequence_mask = train_sequence_mask[train_model_indices][train_model_sequence_available]
    val_model_sequence_mask = train_sequence_mask[val_model_indices][val_model_sequence_available]
    y_train_model_sequence = y_train_model[train_model_sequence_available]
    y_val_model_sequence = y_val_model[val_model_sequence_available]

    X_train_full_sequence = train_sequence_array[train_sequence_available]
    train_full_sequence_mask = train_sequence_mask[train_sequence_available]
    y_train_sequence_full = y_train[train_sequence_available]
    X_test_sequence = test_sequence_array[test_sequence_available]
    test_sequence_mask = test_sequence_mask[test_sequence_available]
    y_test_sequence = y_test[test_sequence_available]
    test_ids_sequence = test_ids.loc[test_sequence_available].reset_index(drop=True)

    sequence_tabular_train_model = X_train_model[train_model_sequence_available]
    sequence_tabular_val_model = X_val_model[val_model_sequence_available]
    sequence_tabular_train_full = X_train[train_sequence_available]
    sequence_tabular_test = X_test[test_sequence_available]

    sequence_coverage_df = pd.DataFrame([
        {"split": "training_model", "admissions": len(y_train_model), "admissions_with_sequence": int(train_model_sequence_available.sum()),
            "coverage_percent": round(100 * train_model_sequence_available.mean(), 2)},
        {"split": "validation", "admissions": len(y_val_model), "admissions_with_sequence": int(val_model_sequence_available.sum()),
            "coverage_percent": round(100 * val_model_sequence_available.mean(), 2)},
        {"split": "full_training", "admissions": len(y_train), "admissions_with_sequence": int(train_sequence_available.sum()),
            "coverage_percent": round(100 * train_sequence_available.mean(), 2)},
        {"split": "test", "admissions": len(y_test), "admissions_with_sequence": int(test_sequence_available.sum()),
            "coverage_percent": round(100 * test_sequence_available.mean(), 2)}])
    sequence_coverage_df["sequence_source"] = sequence_source_label
    #save this table or artefact for later review
    sequence_coverage_df.to_csv(wp3_5_csv_output_path / "t3_5_hourly_clinical_sequence_coverage.csv", index=False)
    pd.DataFrame({"sequence_feature": sequence_feature_columns,
        "source_column": sequence_feature_columns[:len(sequence_raw_columns)] + sequence_raw_columns}).to_csv(
        wp3_5_csv_output_path / "t3_5_hourly_clinical_sequence_features.csv", index=False)
    print("Hourly clinical sequence source:", sequence_source_file)
    print("Hourly clinical sequence coverage:")
    display(sequence_coverage_df)
    print("Sequence input channels:", len(sequence_feature_columns))
    sequence_model_ready = len(y_train_model_sequence) > 0 and len(y_val_model_sequence) > 0 and len(y_test_sequence) > 0



Hourly clinical sequence source: /scratch/DissProject/outputs/t2_3_hourly_clinical_sequence_dataset.csv
Hourly clinical sequence coverage:


,split,admissions,admissions_with_sequence,coverage_percent,sequence_source
0,training_model,162622,24843,15.28,rich_hourly_clinical_sequence
1,validation,28698,4349,15.15,rich_hourly_clinical_sequence
2,full_training,191320,29192,15.26,rich_hourly_clinical_sequence
3,test,47171,7302,15.48,rich_hourly_clinical_sequence


Sequence input channels: 70


In [11]:
#train sequence-only, tabular-only sequence-subset, and hybrid tabular-plus-sequence models
sequence_subset_tabular_performance_df = pd.DataFrame()

if torch_available and run_hourly_clinical_sequence_models and sequence_model_ready:
    sequence_batch_size = 512
    sequence_max_epochs = 50
    sequence_patience = 8
    sequence_learning_rate = 1e-3
    sequence_scale_pos_weight = (y_train_model_sequence == 0).sum() / max((y_train_model_sequence == 1).sum(), 1)

    #define sequence minibatches
    def sequence_minibatches(sequence_array, mask_array, y_array=None, tabular_sparse=None, batch_size=512, shuffle=False, seed=42):
        rng = np.random.default_rng(seed)
        indices = np.arange(sequence_array.shape[0])
        if shuffle:
            rng.shuffle(indices)
        #loop through each item in this feature block
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start:start + batch_size]
            seq_batch = sequence_array[batch_indices].astype(np.float32)
            mask_batch = mask_array[batch_indices].astype(np.float32)
            if tabular_sparse is not None:
                tab_batch = tabular_sparse[batch_indices].toarray().astype(np.float32)
                if y_array is None:
                    yield seq_batch, mask_batch, tab_batch
                else:
                    yield seq_batch, mask_batch, tab_batch, y_array[batch_indices].astype(np.float32)
            else:
                if y_array is None:
                    yield seq_batch, mask_batch
                else:
                    yield seq_batch, mask_batch, y_array[batch_indices].astype(np.float32)

    #define sparse subset minibatches
    def sparse_subset_minibatches(X_sparse, y_array=None, batch_size=1024, shuffle=False, seed=42):
        rng = np.random.default_rng(seed)
        indices = np.arange(X_sparse.shape[0])
        if shuffle:
            rng.shuffle(indices)
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start:start + batch_size]
            X_batch = X_sparse[batch_indices].toarray().astype(np.float32)
            if y_array is None:
                yield X_batch
            else:
                yield X_batch, y_array[batch_indices].astype(np.float32)

    class VitalSequenceGRU(nn.Module):
        #initialise model or helper state
        def __init__(self, input_dim, hidden_dim=64, dropout=0.20):
            super().__init__()
            self.gru = nn.GRU(input_dim, hidden_dim, num_layers=1, batch_first=True)
            self.dropout = nn.Dropout(dropout)
            self.head = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        #define encode
        def encode(self, sequence, mask):
            output, _ = self.gru(sequence)
            lengths = mask.sum(dim=1).long().clamp(min=1)
            last_positions = (lengths - 1).view(-1, 1, 1).expand(-1, 1, output.shape[-1])
            encoded = output.gather(1, last_positions).squeeze(1)
            return self.dropout(encoded)
        #run the model forward pass
        def forward(self, sequence, mask):
            return self.head(self.encode(sequence, mask))

    class VitalSequenceLSTM(nn.Module):
        #initialise model or helper state
        def __init__(self, input_dim, hidden_dim=64, dropout=0.20):
            super().__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=1, batch_first=True)
            self.dropout = nn.Dropout(dropout)
            self.head = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        #define encode
        def encode(self, sequence, mask):
            output, _ = self.lstm(sequence)
            lengths = mask.sum(dim=1).long().clamp(min=1)
            last_positions = (lengths - 1).view(-1, 1, 1).expand(-1, 1, output.shape[-1])
            encoded = output.gather(1, last_positions).squeeze(1)
            return self.dropout(encoded)
        #run the model forward pass
        def forward(self, sequence, mask):
            return self.head(self.encode(sequence, mask))

    class SequenceSubsetTabularMLP(nn.Module):
        #initialise model or helper state
        def __init__(self, input_dim, dropout=0.30):
            super().__init__()
            self.network = nn.Sequential(nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.20), nn.Linear(64, 1))
        #run the model forward pass
        def forward(self, tabular):
            return self.network(tabular)

    class HybridTabularVitalGRU(nn.Module):
        #initialise model or helper state
        def __init__(self, tabular_dim, sequence_dim, hidden_dim=64, dropout=0.25):
            super().__init__()
            self.sequence_encoder = VitalSequenceGRU(sequence_dim, hidden_dim=hidden_dim, dropout=dropout)
            self.tabular_encoder = nn.Sequential(nn.Linear(tabular_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(256, 64), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(hidden_dim + 64, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        #run the model forward pass
        def forward(self, sequence, mask, tabular):
            sequence_embedding = self.sequence_encoder.encode(sequence, mask)
            tabular_embedding = self.tabular_encoder(tabular)
            return self.head(torch.cat([sequence_embedding, tabular_embedding], dim=1))

    class HybridTabularVitalLSTM(nn.Module):
        #initialise model or helper state
        def __init__(self, tabular_dim, sequence_dim, hidden_dim=64, dropout=0.25):
            super().__init__()
            self.sequence_encoder = VitalSequenceLSTM(sequence_dim, hidden_dim=hidden_dim, dropout=dropout)
            self.tabular_encoder = nn.Sequential(nn.Linear(tabular_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(256, 64), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(hidden_dim + 64, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        #run the model forward pass
        def forward(self, sequence, mask, tabular):
            sequence_embedding = self.sequence_encoder.encode(sequence, mask)
            tabular_embedding = self.tabular_encoder(tabular)
            return self.head(torch.cat([sequence_embedding, tabular_embedding], dim=1))

    #predict sequence model
    def predict_sequence_model(model, sequence_array, mask_array, tabular_sparse=None, batch_size=1024):
        model.eval()
        probabilities = []
        with torch.no_grad():
            #loop through each item in this feature block
            for batch in sequence_minibatches(sequence_array, mask_array, tabular_sparse=tabular_sparse, batch_size=batch_size):
                if tabular_sparse is None:
                    seq_batch, mask_batch = batch
                    logits = model(torch.from_numpy(seq_batch).to(device), torch.from_numpy(mask_batch).to(device)).view(-1)
                else:
                    seq_batch, mask_batch, tab_batch = batch
                    logits = model(torch.from_numpy(seq_batch).to(device), torch.from_numpy(mask_batch).to(device),
                        torch.from_numpy(tab_batch).to(device)).view(-1)
                probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())
        return np.concatenate(probabilities)

    #predict sequence subset tabular model
    def predict_sequence_subset_tabular_model(model, X_sparse, batch_size=2048):
        model.eval()
        probabilities = []
        with torch.no_grad():
            for X_batch in sparse_subset_minibatches(X_sparse, batch_size=batch_size):
                logits = model(torch.from_numpy(X_batch).to(device)).view(-1)
                probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())
        return np.concatenate(probabilities)

    #train sequence model
    def train_sequence_model(model, model_name, tabular_train=None, tabular_val=None):
        model = model.to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([sequence_scale_pos_weight], dtype=torch.float32, device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=sequence_learning_rate, weight_decay=1e-4)
        best_state = None
        best_val_ap = -np.inf
        stale_epochs = 0
        history_rows = []
        training_start = time.perf_counter()
        for epoch in range(1, sequence_max_epochs + 1):
            model.train()
            epoch_losses = []
            for batch in sequence_minibatches(X_train_model_sequence, train_model_sequence_mask, y_train_model_sequence,
                    tabular_sparse=tabular_train, batch_size=sequence_batch_size, shuffle=True, seed=500 + epoch):
                optimizer.zero_grad()
                if tabular_train is None:
                    seq_batch, mask_batch, y_batch = batch
                    logits = model(torch.from_numpy(seq_batch).to(device), torch.from_numpy(mask_batch).to(device)).view(-1)
                else:
                    seq_batch, mask_batch, tab_batch, y_batch = batch
                    logits = model(torch.from_numpy(seq_batch).to(device), torch.from_numpy(mask_batch).to(device),
                        torch.from_numpy(tab_batch).to(device)).view(-1)
                loss = criterion(logits, torch.from_numpy(y_batch).to(device))
                loss.backward()
                optimizer.step()
                epoch_losses.append(float(loss.detach().cpu().item()))
            val_probability = predict_sequence_model(model, X_val_model_sequence, val_model_sequence_mask, tabular_sparse=tabular_val)
            val_ap = average_precision_score(y_val_model_sequence, val_probability)
            val_roc = roc_auc_score(y_val_model_sequence, val_probability)
            history_rows.append({"epoch": epoch, "training_loss": np.mean(epoch_losses),
                "validation_pr_auc_average_precision": val_ap, "validation_roc_auc": val_roc})
            print(model_name, "epoch", epoch, "loss", round(np.mean(epoch_losses), 4),
                "val PR-AUC", round(val_ap, 4), "val ROC-AUC", round(val_roc, 4))
            if val_ap > best_val_ap + 1e-5:
                best_val_ap = val_ap
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
                if stale_epochs >= sequence_patience:
                    print("Early stopping after", epoch, "epochs.")
                    break
        training_time_seconds = time.perf_counter() - training_start
        if best_state is not None:
            model.load_state_dict(best_state)
            model = model.to(device)
        return model, pd.DataFrame(history_rows), training_time_seconds

    #train sequence subset tabular model
    def train_sequence_subset_tabular_model(model, model_name):
        model = model.to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([sequence_scale_pos_weight], dtype=torch.float32, device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=3e-4)
        best_state = None
        best_val_ap = -np.inf
        stale_epochs = 0
        history_rows = []
        training_start = time.perf_counter()
        for epoch in range(1, 50 + 1):
            model.train()
            epoch_losses = []
            for tab_batch, y_batch in sparse_subset_minibatches(sequence_tabular_train_model, y_train_model_sequence,
                    batch_size=1024, shuffle=True, seed=700 + epoch):
                optimizer.zero_grad()
                logits = model(torch.from_numpy(tab_batch).to(device)).view(-1)
                loss = criterion(logits, torch.from_numpy(y_batch).to(device))
                loss.backward()
                optimizer.step()
                epoch_losses.append(float(loss.detach().cpu().item()))
            val_probability = predict_sequence_subset_tabular_model(model, sequence_tabular_val_model)
            val_ap = average_precision_score(y_val_model_sequence, val_probability)
            val_roc = roc_auc_score(y_val_model_sequence, val_probability)
            history_rows.append({"epoch": epoch, "training_loss": np.mean(epoch_losses),
                "validation_pr_auc_average_precision": val_ap, "validation_roc_auc": val_roc})
            print(model_name, "epoch", epoch, "loss", round(np.mean(epoch_losses), 4),
                "val PR-AUC", round(val_ap, 4), "val ROC-AUC", round(val_roc, 4))
            if val_ap > best_val_ap + 1e-5:
                best_val_ap = val_ap
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
                if stale_epochs >= 7:
                    print("Early stopping after", epoch, "epochs.")
                    break
        training_time_seconds = time.perf_counter() - training_start
        if best_state is not None:
            model.load_state_dict(best_state)
            model = model.to(device)
        return model, pd.DataFrame(history_rows), training_time_seconds

    #save sequence outputs
    def save_sequence_outputs(model_name, output_stem, train_probability, val_probability, test_probability,
            training_time_seconds, train_inference_time_seconds, test_inference_time_seconds):
        f1_threshold, balanced_pr_threshold, threshold_df = find_thresholds(y_val_model_sequence, val_probability)
        #save this table or artefact for later review
        threshold_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_validation_threshold_search.csv", index=False)
        rows = []
        for threshold_role, threshold in [("default", 0.50), ("validation_f1", f1_threshold), ("balanced_precision_recall", balanced_pr_threshold)]:
            row = evaluate_probability_model(model_name, y_test_sequence, test_probability, threshold,
                training_time_seconds, train_inference_time_seconds, test_inference_time_seconds)
            row["threshold_role"] = threshold_role
            row["evaluation_subset"] = "admissions_with_first72h_hourly_clinical_sequence"
            row["subset_admissions"] = len(y_test_sequence)
            rows.append(row)
        performance_df = pd.DataFrame(rows)
        performance_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_performance.csv", index=False)
        prediction_df = test_ids_sequence.copy()
        prediction_df["predicted_probability"] = test_probability
        prediction_df["predicted_class"] = (test_probability >= f1_threshold).astype(int)
        prediction_df.to_csv(wp3_5_csv_output_path / f"{output_stem}_test_predictions.csv", index=False)
        print(model_name, "sequence-subset test PR-AUC:", round(average_precision_score(y_test_sequence, test_probability), 4),
            "ROC-AUC:", round(roc_auc_score(y_test_sequence, test_probability), 4), "validation F1 threshold:", f1_threshold)
        return performance_df, threshold_df, prediction_df

    #this tabular-only row is the fair comparator for judging whether hourly sequence data adds value.
    sequence_subset_tabular_model = SequenceSubsetTabularMLP(X_train.shape[1])
    sequence_subset_tabular_model, sequence_subset_tabular_history_df, sequence_subset_tabular_training_time_seconds = train_sequence_subset_tabular_model(
        sequence_subset_tabular_model, "Sequence-subset Tabular MLP")
    #save this table or artefact for later review
    sequence_subset_tabular_history_df.to_csv(wp3_5_csv_output_path / "t3_5_sequence_subset_tabular_mlp_training_history.csv", index=False)
    torch.save(sequence_subset_tabular_model.state_dict(), wp3_5_model_output_path / "t3_5_sequence_subset_tabular_mlp_state_dict.pt")
    sequence_tabular_train_start = time.perf_counter()
    sequence_subset_tabular_train_probability = predict_sequence_subset_tabular_model(sequence_subset_tabular_model, sequence_tabular_train_full)
    sequence_subset_tabular_train_inference_time_seconds = time.perf_counter() - sequence_tabular_train_start
    sequence_subset_tabular_val_probability = predict_sequence_subset_tabular_model(sequence_subset_tabular_model, sequence_tabular_val_model)
    sequence_tabular_test_start = time.perf_counter()
    sequence_subset_tabular_test_probability = predict_sequence_subset_tabular_model(sequence_subset_tabular_model, sequence_tabular_test)
    sequence_subset_tabular_test_inference_time_seconds = time.perf_counter() - sequence_tabular_test_start
    sequence_subset_tabular_performance_df, _, _ = save_sequence_outputs("Sequence-subset Tabular MLP", "t3_5_sequence_subset_tabular_mlp",
        sequence_subset_tabular_train_probability, sequence_subset_tabular_val_probability, sequence_subset_tabular_test_probability,
        sequence_subset_tabular_training_time_seconds, sequence_subset_tabular_train_inference_time_seconds,
        sequence_subset_tabular_test_inference_time_seconds)

    sequence_only_performance_frames = []
    hybrid_sequence_performance_frames = []

    vital_sequence_model = VitalSequenceGRU(input_dim=len(sequence_feature_columns))
    vital_sequence_model, vital_sequence_history_df, vital_sequence_training_time_seconds = train_sequence_model(
        vital_sequence_model, "Hourly Clinical GRU")
    vital_sequence_history_df.to_csv(wp3_5_csv_output_path / "t3_5_hourly_clinical_gru_training_history.csv", index=False)
    torch.save(vital_sequence_model.state_dict(), wp3_5_model_output_path / "t3_5_hourly_clinical_gru_state_dict.pt")
    seq_train_start = time.perf_counter()
    vital_sequence_train_probability = predict_sequence_model(vital_sequence_model, X_train_full_sequence, train_full_sequence_mask)
    vital_sequence_train_inference_time_seconds = time.perf_counter() - seq_train_start
    vital_sequence_val_probability = predict_sequence_model(vital_sequence_model, X_val_model_sequence, val_model_sequence_mask)
    seq_test_start = time.perf_counter()
    vital_sequence_test_probability = predict_sequence_model(vital_sequence_model, X_test_sequence, test_sequence_mask)
    vital_sequence_test_inference_time_seconds = time.perf_counter() - seq_test_start
    vital_sequence_performance_df, _, _ = save_sequence_outputs("Hourly Clinical GRU", "t3_5_hourly_clinical_gru",
        vital_sequence_train_probability, vital_sequence_val_probability, vital_sequence_test_probability,
        vital_sequence_training_time_seconds, vital_sequence_train_inference_time_seconds, vital_sequence_test_inference_time_seconds)
    sequence_only_performance_frames.append(vital_sequence_performance_df)

    lstm_sequence_model = VitalSequenceLSTM(input_dim=len(sequence_feature_columns))
    lstm_sequence_model, lstm_sequence_history_df, lstm_sequence_training_time_seconds = train_sequence_model(
        lstm_sequence_model, "Hourly Clinical LSTM")
    lstm_sequence_history_df.to_csv(wp3_5_csv_output_path / "t3_5_hourly_clinical_lstm_training_history.csv", index=False)
    torch.save(lstm_sequence_model.state_dict(), wp3_5_model_output_path / "t3_5_hourly_clinical_lstm_state_dict.pt")
    lstm_train_start = time.perf_counter()
    lstm_sequence_train_probability = predict_sequence_model(lstm_sequence_model, X_train_full_sequence, train_full_sequence_mask)
    lstm_sequence_train_inference_time_seconds = time.perf_counter() - lstm_train_start
    lstm_sequence_val_probability = predict_sequence_model(lstm_sequence_model, X_val_model_sequence, val_model_sequence_mask)
    lstm_test_start = time.perf_counter()
    lstm_sequence_test_probability = predict_sequence_model(lstm_sequence_model, X_test_sequence, test_sequence_mask)
    lstm_sequence_test_inference_time_seconds = time.perf_counter() - lstm_test_start
    lstm_sequence_performance_df, _, _ = save_sequence_outputs("Hourly Clinical LSTM", "t3_5_hourly_clinical_lstm",
        lstm_sequence_train_probability, lstm_sequence_val_probability, lstm_sequence_test_probability,
        lstm_sequence_training_time_seconds, lstm_sequence_train_inference_time_seconds, lstm_sequence_test_inference_time_seconds)
    sequence_only_performance_frames.append(lstm_sequence_performance_df)

    hybrid_sequence_model = HybridTabularVitalGRU(tabular_dim=X_train.shape[1], sequence_dim=len(sequence_feature_columns))
    hybrid_sequence_model, hybrid_sequence_history_df, hybrid_sequence_training_time_seconds = train_sequence_model(
        hybrid_sequence_model, "Hybrid Tabular + Hourly Clinical GRU", tabular_train=sequence_tabular_train_model,
        tabular_val=sequence_tabular_val_model)
    hybrid_sequence_history_df.to_csv(wp3_5_csv_output_path / "t3_5_hybrid_tabular_hourly_clinical_gru_training_history.csv", index=False)
    torch.save(hybrid_sequence_model.state_dict(), wp3_5_model_output_path / "t3_5_hybrid_tabular_hourly_clinical_gru_state_dict.pt")
    hybrid_train_start = time.perf_counter()
    hybrid_sequence_train_probability = predict_sequence_model(hybrid_sequence_model, X_train_full_sequence, train_full_sequence_mask,
        tabular_sparse=sequence_tabular_train_full)
    hybrid_sequence_train_inference_time_seconds = time.perf_counter() - hybrid_train_start
    hybrid_sequence_val_probability = predict_sequence_model(hybrid_sequence_model, X_val_model_sequence, val_model_sequence_mask,
        tabular_sparse=sequence_tabular_val_model)
    hybrid_test_start = time.perf_counter()
    hybrid_sequence_test_probability = predict_sequence_model(hybrid_sequence_model, X_test_sequence, test_sequence_mask,
        tabular_sparse=sequence_tabular_test)
    hybrid_sequence_test_inference_time_seconds = time.perf_counter() - hybrid_test_start
    hybrid_gru_performance_df, _, _ = save_sequence_outputs("Hybrid Tabular + Hourly Clinical GRU",
        "t3_5_hybrid_tabular_hourly_clinical_gru", hybrid_sequence_train_probability, hybrid_sequence_val_probability,
        hybrid_sequence_test_probability, hybrid_sequence_training_time_seconds, hybrid_sequence_train_inference_time_seconds,
        hybrid_sequence_test_inference_time_seconds)
    hybrid_sequence_performance_frames.append(hybrid_gru_performance_df)

    hybrid_lstm_sequence_model = HybridTabularVitalLSTM(tabular_dim=X_train.shape[1], sequence_dim=len(sequence_feature_columns))
    hybrid_lstm_sequence_model, hybrid_lstm_sequence_history_df, hybrid_lstm_sequence_training_time_seconds = train_sequence_model(
        hybrid_lstm_sequence_model, "Hybrid Tabular + Hourly Clinical LSTM", tabular_train=sequence_tabular_train_model,
        tabular_val=sequence_tabular_val_model)
    hybrid_lstm_sequence_history_df.to_csv(wp3_5_csv_output_path / "t3_5_hybrid_tabular_hourly_clinical_lstm_training_history.csv", index=False)
    torch.save(hybrid_lstm_sequence_model.state_dict(), wp3_5_model_output_path / "t3_5_hybrid_tabular_hourly_clinical_lstm_state_dict.pt")
    hybrid_lstm_train_start = time.perf_counter()
    hybrid_lstm_sequence_train_probability = predict_sequence_model(hybrid_lstm_sequence_model, X_train_full_sequence, train_full_sequence_mask,
        tabular_sparse=sequence_tabular_train_full)
    hybrid_lstm_sequence_train_inference_time_seconds = time.perf_counter() - hybrid_lstm_train_start
    hybrid_lstm_sequence_val_probability = predict_sequence_model(hybrid_lstm_sequence_model, X_val_model_sequence, val_model_sequence_mask,
        tabular_sparse=sequence_tabular_val_model)
    hybrid_lstm_test_start = time.perf_counter()
    hybrid_lstm_sequence_test_probability = predict_sequence_model(hybrid_lstm_sequence_model, X_test_sequence, test_sequence_mask,
        tabular_sparse=sequence_tabular_test)
    hybrid_lstm_sequence_test_inference_time_seconds = time.perf_counter() - hybrid_lstm_test_start
    hybrid_lstm_performance_df, _, _ = save_sequence_outputs("Hybrid Tabular + Hourly Clinical LSTM",
        "t3_5_hybrid_tabular_hourly_clinical_lstm", hybrid_lstm_sequence_train_probability, hybrid_lstm_sequence_val_probability,
        hybrid_lstm_sequence_test_probability, hybrid_lstm_sequence_training_time_seconds,
        hybrid_lstm_sequence_train_inference_time_seconds, hybrid_lstm_sequence_test_inference_time_seconds)
    hybrid_sequence_performance_frames.append(hybrid_lstm_performance_df)

    hourly_sequence_performance_df = pd.concat(sequence_only_performance_frames, ignore_index=True)
    hybrid_sequence_performance_df = pd.concat(hybrid_sequence_performance_frames, ignore_index=True)
else:
    print("Hourly clinical sequence models were not trained because the sequence data were unavailable or PyTorch was skipped.")

Sequence-subset Tabular MLP epoch 1 loss 1.0548 val PR-AUC 0.4041 val ROC-AUC 0.7357
Sequence-subset Tabular MLP epoch 2 loss 0.9642 val PR-AUC 0.4251 val ROC-AUC 0.7504
Sequence-subset Tabular MLP epoch 3 loss 0.9153 val PR-AUC 0.4298 val ROC-AUC 0.7473
Sequence-subset Tabular MLP epoch 4 loss 0.8777 val PR-AUC 0.435 val ROC-AUC 0.756
Sequence-subset Tabular MLP epoch 5 loss 0.8615 val PR-AUC 0.4344 val ROC-AUC 0.7518
Sequence-subset Tabular MLP epoch 6 loss 0.8215 val PR-AUC 0.4291 val ROC-AUC 0.7416
Sequence-subset Tabular MLP epoch 7 loss 0.7917 val PR-AUC 0.4228 val ROC-AUC 0.7511
Sequence-subset Tabular MLP epoch 8 loss 0.7559 val PR-AUC 0.4226 val ROC-AUC 0.7495
Sequence-subset Tabular MLP epoch 9 loss 0.7385 val PR-AUC 0.4077 val ROC-AUC 0.739
Sequence-subset Tabular MLP epoch 10 loss 0.706 val PR-AUC 0.4156 val ROC-AUC 0.7464
Sequence-subset Tabular MLP epoch 11 loss 0.672 val PR-AUC 0.4171 val ROC-AUC 0.7417
Early stopping after 11 epochs.
Sequence-subset Tabular MLP sequence

## 3.5.9 Neural Model Summary

This final section combines any neural-network models that were successfully trained. These outputs can be read by WP3.6 alongside the logistic, random forest, and boosting model outputs.





In [12]:
#combine neural-network comparator outputs
neural_performance_frames = [df for df in [mlp_performance_df, tabtransformer_performance_df, tabnet_performance_df,
        sequence_subset_tabular_performance_df, hourly_sequence_performance_df, hybrid_sequence_performance_df]
    if isinstance(df, pd.DataFrame) and not df.empty]
if neural_performance_frames:
    neural_model_comparison_df = pd.concat(neural_performance_frames, ignore_index=True)
    neural_model_comparison_df.to_csv(wp3_5_csv_output_path / "t3_5_neural_model_comparison.csv", index=False)
    neural_model_comparison_df.to_csv(wp3_5_output_path / "t3_5_neural_model_comparison.csv", index=False)
    display(neural_model_comparison_df.sort_values(["pr_auc_average_precision", "roc_auc"], ascending=False).round(4))
else:
    neural_model_comparison_df = pd.DataFrame()
    print("No neural-network models were trained in this run.")

summary_rows = []
for file_path in sorted(wp3_5_output_path.rglob("*")):
    if file_path.is_file():
        summary_rows.append({"output_folder": str(file_path.parent.relative_to(wp3_5_output_path)),
            "file_name": file_path.name, "file_size_mb": round(file_path.stat().st_size / (1024 ** 2), 3)})
wp3_5_output_file_summary_df = pd.DataFrame(summary_rows)
wp3_5_output_file_summary_df.to_csv(wp3_5_csv_output_path / "t3_5_neural_output_file_summary.csv", index=False)
print("WP3.5 neural-network model notebook complete. Outputs saved to:")
print(wp3_5_output_path)

,model,threshold,accuracy,balanced_accuracy,precision,recall_sensitivity,specificity,f1_score,roc_auc,pr_auc_average_precision,...,false_positives,false_negatives,true_positives,training_time_seconds,train_inference_time_seconds,test_inference_time_seconds,test_inference_time_per_admission_ms,threshold_role,evaluation_subset,subset_admissions
0,PyTorch MLP - mlp_screen_06_384-192-64_wd0p000...,0.50,0.7525,0.7107,0.4177,0.6417,0.7797,0.5060,0.7911,0.5807,...,8337,3339,5980,7.4692,0.6465,0.1622,0.0034,default,NaN,NaN
1,PyTorch MLP - mlp_screen_06_384-192-64_wd0p000...,0.59,0.8106,0.7011,0.5208,0.5199,0.8822,0.5204,0.7911,0.5807,...,4458,4474,4845,7.4692,0.6465,0.1622,0.0034,validation_f1,NaN,NaN
2,PyTorch MLP - mlp_screen_06_384-192-64_wd0p000...,0.60,0.8152,0.6993,0.5340,0.5076,0.8910,0.5205,0.7911,0.5807,...,4127,4589,4730,7.4692,0.6465,0.1622,0.0034,balanced_precision_recall,NaN,NaN
3,Processed-feature TabTransformer - tabtransfor...,0.50,0.7503,0.7095,0.4148,0.6420,0.7770,0.5040,0.7879,0.5722,...,8441,3336,5983,404.2513,3.2003,0.7877,0.0167,default,NaN,NaN
4,Processed-feature TabTransformer - tabtransfor...,0.59,0.8025,0.7041,0.5000,0.5414,0.8667,0.5199,0.7879,0.5722,...,5044,4274,5045,404.2513,3.2003,0.7877,0.0167,validation_f1,NaN,NaN
5,Processed-feature TabTransformer - tabtransfor...,0.61,0.8105,0.7001,0.5204,0.5175,0.8826,0.5190,0.7879,0.5722,...,4445,4496,4823,404.2513,3.2003,0.7877,0.0167,balanced_precision_recall,NaN,NaN
6,TabNet - tabnet_screen_05_d16_steps4_gamma1p6_...,0.50,0.7508,0.7088,0.4152,0.6394,0.7782,0.5035,0.7844,0.5698,...,8394,3360,5959,377.8978,1.6585,0.4085,0.0087,default,NaN,NaN
7,TabNet - tabnet_screen_05_d16_steps4_gamma1p6_...,0.60,0.8048,0.6952,0.5058,0.5141,0.8763,0.5099,0.7844,0.5698,...,4682,4528,4791,377.8978,1.6585,0.4085,0.0087,validation_f1,NaN,NaN
8,TabNet - tabnet_screen_05_d16_steps4_gamma1p6_...,0.61,0.8091,0.6931,0.5174,0.5014,0.8848,0.5093,0.7844,0.5698,...,4359,4646,4673,377.8978,1.6585,0.4085,0.0087,balanced_precision_recall,NaN,NaN
9,Sequence-subset Tabular MLP,0.50,0.7223,0.6981,0.3115,0.6631,0.7330,0.4239,0.7729,0.4432,...,1649,379,746,1.0116,0.0748,0.0191,0.0026,default,admissions_with_first72h_hourly_clinical_sequence,7302.0


WP3.5 neural-network model notebook complete. Outputs saved to:
/scratch/DissProject/outputs/WP3_5


**Consolidated CSV Table Workbook**

This cell keeps the individual CSV files and also creates one Excel workbook where each CSV table is available as a separate sheet. The workbook is for easier review and sharing; the CSV files remain the primary machine-readable outputs.


In [13]:
#compile csv tables into one workbook while preserving the original csv files
from collections import Counter
import re

possible_output_roots = []
#loop through each item in this feature block
for output_name in ['wp3_5_output_path']:
    if output_name in globals():
        output_root = globals()[output_name]
        if isinstance(output_root, Path) and output_root.exists() and output_root not in possible_output_roots:
            possible_output_roots.append(output_root)

if not possible_output_roots:
    print("no wp3 output roots were available for workbook compilation")
else:
    primary_output_root = possible_output_roots[0]
    workbook_path = primary_output_root / f"{primary_output_root.name.lower()}_csv_table_workbook.xlsx"
    csv_paths = []
    #loop through each item in this feature block
    for output_root in possible_output_roots:
        csv_paths.extend(sorted(path for path in output_root.rglob("*.csv") if path.is_file()))
    csv_paths = sorted(dict.fromkeys(csv_paths))

    #make sheet name
    def make_sheet_name(csv_path, used_names):
        relative_name = csv_path.relative_to(primary_output_root.parent).with_suffix("").as_posix()
        relative_name = re.sub(r"[^0-9a-zA-Z]+", "_", relative_name).strip("_").lower()
        parts = [part for part in relative_name.split("_") if part not in ["outputs", primary_output_root.name.lower(), "csv", "tables"]]
        base_name = "_".join(parts[-4:]) if parts else csv_path.stem.lower()
        base_name = base_name[:31] or "table"
        sheet_name = base_name
        counter = 1
        while sheet_name in used_names:
            suffix = f"_{counter}"
            sheet_name = base_name[:31 - len(suffix)] + suffix
            counter += 1
        used_names.add(sheet_name)
        return sheet_name

    workbook_index_rows = []
    used_sheet_names = set()
    max_excel_rows = 200000
    with pd.ExcelWriter(workbook_path) as writer:
        #loop through each item in this feature block
        for csv_path in csv_paths:
            try:
                table_df = pd.read_csv(csv_path)
            except Exception as error:
                workbook_index_rows.append({"csv_path": str(csv_path), "sheet_name": "", "rows": np.nan,
                    "columns": np.nan, "rows_written": 0, "note": f"read_failed: {error}"})
                continue
            sheet_name = make_sheet_name(csv_path, used_sheet_names)
            rows_written = min(len(table_df), max_excel_rows)
            #save this table or artefact for later review
            table_df.head(max_excel_rows).to_excel(writer, sheet_name=sheet_name, index=False)
            workbook_index_rows.append({"csv_path": str(csv_path), "sheet_name": sheet_name, "rows": len(table_df),
                "columns": table_df.shape[1], "rows_written": rows_written,
                "note": "truncated_for_excel" if len(table_df) > max_excel_rows else "complete"})
        workbook_index_df = pd.DataFrame(workbook_index_rows)
        #save this table or artefact for later review
        workbook_index_df.to_excel(writer, sheet_name="workbook_index", index=False)
    #save this table or artefact for later review
    workbook_index_df.to_csv(primary_output_root / f"{primary_output_root.name.lower()}_csv_table_workbook_index.csv", index=False)
    print("csv table workbook saved to:")
    print(workbook_path)
    display(workbook_index_df)

csv table workbook saved to:
/scratch/DissProject/outputs/WP3_5/wp3_5_csv_table_workbook.xlsx


,csv_path,sheet_name,rows,columns,rows_written,note
0,/scratch/DissProject/outputs/WP3_5/csv_tables/...,hourly_clinical_gru_performance,3,22,3,complete
1,/scratch/DissProject/outputs/WP3_5/csv_tables/...,clinical_gru_test_predictions,7302,15,7302,complete
2,/scratch/DissProject/outputs/WP3_5/csv_tables/...,clinical_gru_training_history,13,4,13,complete
3,/scratch/DissProject/outputs/WP3_5/csv_tables/...,gru_validation_threshold_search,99,7,99,complete
4,/scratch/DissProject/outputs/WP3_5/csv_tables/...,hourly_clinical_lstm_performanc,3,22,3,complete
...,...,...,...,...,...,...
181,/scratch/DissProject/outputs/WP3_5/csv_tables/...,tabtransformer_style_training_h,39,11,39,complete
182,/scratch/DissProject/outputs/WP3_5/csv_tables/...,style_validation_threshold_sear,99,7,99,complete
183,/scratch/DissProject/outputs/WP3_5/csv_tables/...,tabtransformer_style_variant_su,8,14,8,complete
184,/scratch/DissProject/outputs/WP3_5/t3_5_neural...,5_neural_model_comparison_1,24,22,24,complete
